In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
STORAGE_ACCOUNT = "adlsairbnbde" 
spark.conf.set(
    "fs.azure.account.key.adlsairbnbde.dfs.core.windows.net",
    "KEY HERE"
)
spark.conf.set("spark.databricks.delta.properties.defaults.enableDeletionVectors", "false")
GOLD_BASE = f"abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net"
FACT_LISTING_SNAPSHOT_PATH = f"{GOLD_BASE}/fact_listing_snapshot"
DIM_HOST_PATH = f"{GOLD_BASE}/dim_host"
DIM_LISTING_PATH = f"{GOLD_BASE}/dim_listing"

In [0]:
fact_listing_snapshot = spark.read.format("delta").load(FACT_LISTING_SNAPSHOT_PATH)

quarters_in_order = [row.quarter_label for row in
                      fact_listing_snapshot.select("quarter_label").distinct()
                      .orderBy("quarter_label").collect()]
 
print(quarters_in_order)

['2025-Q3', '2026-Q2']


In [0]:
def get_host_snapshot(quarter_label):
    return (
        fact_listing_snapshot
        .filter(F.col("quarter_label") == quarter_label)
        .select("host_id", "city", "host_is_superhost", "host_listings_count")
        .dropDuplicates(["host_id", "city"])
    )
 
def get_listing_snapshot(quarter_label):
    return (
        fact_listing_snapshot
        .filter(F.col("quarter_label") == quarter_label)
        .select("listing_id", "city", "neighbourhood_key", "room_type", "accommodates", "bedrooms")
        .dropDuplicates(["listing_id", "city"])
    )

In [0]:

def process_scd2(table_path, business_keys, tracked_columns, new_snapshot_df, quarter_label, table_name):
    new_snapshot_df = (
        new_snapshot_df
        .withColumn("effective_start_quarter", F.lit(quarter_label))
        .withColumn("effective_end_quarter", F.lit(None).cast("string"))
        .withColumn("is_current", F.lit(True))
    )
 
    if not DeltaTable.isDeltaTable(spark, table_path):
        print(f"[{table_name}] No existing table found — initializing with quarter '{quarter_label}'.")
        new_snapshot_df.write.format("delta").mode("overwrite").save(table_path)
        print(f"[{table_name}] Initialized with {new_snapshot_df.count()} rows.")
        return
 
    dim_table = DeltaTable.forPath(spark, table_path)
    current_rows_df = dim_table.toDF().filter(F.col("is_current") == True)
 
    # Entities whose tracked attributes differ from the current record
    change_filter = " OR ".join([f"current.{c} != new.{c}" for c in tracked_columns])
    changed_keys_df = (
        current_rows_df.alias("current")
        .join(new_snapshot_df.alias("new"), on=business_keys, how="inner")
        .filter(change_filter)
        .select(*[F.col(f"new.{k}").alias(k) for k in business_keys])
    )
    changed_count = changed_keys_df.count()
    print(f"[{table_name}] Entities with changed attributes this quarter: {changed_count}")
 
    if changed_count > 0:
        merge_condition = " AND ".join([f"target.{k} = source.{k}" for k in business_keys]) + " AND target.is_current = true"
        (
            dim_table.alias("target")
            .merge(changed_keys_df.alias("source"), merge_condition)
            .whenMatchedUpdate(set={
                "is_current": F.lit(False),
                "effective_end_quarter": F.lit(quarter_label),
            })
            .execute()
        )
        print(f"[{table_name}] Closed out {changed_count} changed current row(s).")
 
    # Brand-new entities: appear in the new snapshot but have no existing row at all
    existing_keys_df = current_rows_df.select(*business_keys).distinct()
    brand_new_df = new_snapshot_df.join(existing_keys_df, on=business_keys, how="left_anti")
    brand_new_count = brand_new_df.count()
 
    # Changed entities need a fresh "current" row inserted too
    changed_new_rows_df = (
        new_snapshot_df.join(changed_keys_df, on=business_keys, how="inner")
        if changed_count > 0 else new_snapshot_df.limit(0)
    )
 
    rows_to_insert = brand_new_df.unionByName(changed_new_rows_df)
    rows_to_insert_count = rows_to_insert.count()
 
    if rows_to_insert_count > 0:
        rows_to_insert.write.format("delta").mode("append").save(table_path)
        print(
            f"[{table_name}] Inserted {rows_to_insert_count} new current row(s) "
            f"({brand_new_count} brand-new, {changed_count} due to attribute changes)."
        )
    else:
        print(f"[{table_name}] No new rows to insert this quarter — fully idempotent, nothing changed.")

In [0]:
HOST_BUSINESS_KEYS = ["host_id", "city"]
HOST_TRACKED_COLUMNS = ["host_is_superhost", "host_listings_count"]
 
for quarter in quarters_in_order:
    print(f"\n=== dim_host: processing {quarter} ===")
    snapshot = get_host_snapshot(quarter)
    process_scd2(DIM_HOST_PATH, HOST_BUSINESS_KEYS, HOST_TRACKED_COLUMNS, snapshot, quarter, "dim_host")


=== dim_host: processing 2025-Q3 ===
[dim_host] No existing table found — initializing with quarter '2025-Q3'.
[dim_host] Initialized with 16486 rows.

=== dim_host: processing 2026-Q2 ===
[dim_host] Entities with changed attributes this quarter: 3936
[dim_host] Closed out 3936 changed current row(s).
[dim_host] Inserted 5726 new current row(s) (5726 brand-new, 3936 due to attribute changes).


In [0]:
LISTING_BUSINESS_KEYS = ["listing_id", "city"]
LISTING_TRACKED_COLUMNS = ["neighbourhood_key", "room_type", "accommodates", "bedrooms"]
 
for quarter in quarters_in_order:
    print(f"\n=== dim_listing: processing {quarter} ===")
    snapshot = get_listing_snapshot(quarter)
    process_scd2(DIM_LISTING_PATH, LISTING_BUSINESS_KEYS, LISTING_TRACKED_COLUMNS, snapshot, quarter, "dim_listing")


=== dim_listing: processing 2025-Q3 ===
[dim_listing] No existing table found — initializing with quarter '2025-Q3'.
[dim_listing] Initialized with 44859 rows.

=== dim_listing: processing 2026-Q2 ===
[dim_listing] Entities with changed attributes this quarter: 2245
[dim_listing] Closed out 2245 changed current row(s).
[dim_listing] Inserted 9560 new current row(s) (9560 brand-new, 2245 due to attribute changes).


In [0]:
dbutils.fs.rm(DIM_HOST_PATH, recurse=True)
dbutils.fs.rm(DIM_LISTING_PATH, recurse=True)
print("Old dim_host and dim_listing folders deleted.")

Old dim_host and dim_listing folders deleted.


In [0]:

dim_host_final = spark.read.format("delta").load(DIM_HOST_PATH)
 
lost_superhost = (
    dim_host_final.alias("old")
    .join(
        dim_host_final.alias("new"),
        (F.col("old.host_id") == F.col("new.host_id")) &
        (F.col("old.city") == F.col("new.city")) &
        (F.col("old.is_current") == False) &
        (F.col("new.is_current") == True),
        "inner"
    )
    .filter(
        (F.col("old.host_is_superhost") == True) &
        (F.col("new.host_is_superhost") == False)
    )
    .select(
        F.col("old.host_id"),
        F.col("old.city"),
        F.col("old.effective_start_quarter").alias("was_superhost_as_of"),
        F.col("new.effective_start_quarter").alias("lost_status_as_of"),
    )
)
 
print(f"Hosts who LOST superhost status: {lost_superhost.count()}")
display(lost_superhost)

Hosts who LOST superhost status: 910


host_id,city,was_superhost_as_of,lost_status_as_of
917871,lisbon,2025-Q3,2026-Q2
3878913,lisbon,2025-Q3,2026-Q2
498720237,lisbon,2025-Q3,2026-Q2
513927952,lisbon,2025-Q3,2026-Q2
192909198,lisbon,2025-Q3,2026-Q2
551120599,lisbon,2025-Q3,2026-Q2
26918474,lisbon,2025-Q3,2026-Q2
15847122,lisbon,2025-Q3,2026-Q2
659895210,lisbon,2025-Q3,2026-Q2
136971075,lisbon,2025-Q3,2026-Q2


In [0]:

gained_superhost = (
    dim_host_final.alias("old")
    .join(
        dim_host_final.alias("new"),
        (F.col("old.host_id") == F.col("new.host_id")) &
        (F.col("old.city") == F.col("new.city")) &
        (F.col("old.is_current") == False) &
        (F.col("new.is_current") == True),
        "inner"
    )
    .filter(
        (F.col("old.host_is_superhost") == False) &
        (F.col("new.host_is_superhost") == True)
    )
    .select(
        F.col("old.host_id"),
        F.col("old.city"),
        F.col("new.effective_start_quarter").alias("gained_status_as_of"),
    )
)
 
print(f"Hosts who GAINED superhost status: {gained_superhost.count()}")
display(gained_superhost)

Hosts who GAINED superhost status: 1045


host_id,city,gained_status_as_of
506411126,lisbon,2026-Q2
22429260,lisbon,2026-Q2
9686637,lisbon,2026-Q2
39370045,lisbon,2026-Q2
2062006,lisbon,2026-Q2
181414151,lisbon,2026-Q2
533094722,lisbon,2026-Q2
673104063,lisbon,2026-Q2
28296743,lisbon,2026-Q2
130473109,lisbon,2026-Q2


In [0]:
dim_host_final_count = spark.read.format("delta").load(DIM_HOST_PATH).count()
dim_listing_final_count = spark.read.format("delta").load(DIM_LISTING_PATH).count()
 

print(f"  dim_host total rows (including history):    {dim_host_final_count}")
print(f"  dim_listing total rows (including history): {dim_listing_final_count}")
print(f"  Hosts who lost superhost status:            {lost_superhost.count()}")
print(f"  Hosts who gained superhost status:           {gained_superhost.count()}")

  dim_host total rows (including history):    22212
  dim_listing total rows (including history): 54419
  Hosts who lost superhost status:            910
  Hosts who gained superhost status:           1045
